# Boundary layer on a rotating disk - von Karman flow

This notebook is used for assessing the accuracy of the numerical setup (B.C and I.C.) for the von Karman flow field over a rotating disk. We consider a steady, laminar, axially-symmetric viscous flow induced by an infinte disk rotating steadily with angular velocity $\Omega$ about the $z$-axis in a cylindrical coordinate system $(r, \theta, z)$. The motion of the incompressible viscous fluid is, thus, governed by 

>$$  \begin{align*} 
U\frac{\partial U}{\partial r} + W\frac{\partial U}{\partial z} + \frac{V^2}{r} 
&= - \frac{1}{\rho} \frac{\partial P}{\partial r} + \nu \left( \frac{\partial^2 U}{\partial r^2} + \frac{1}{r}\frac{\partial U}{\partial r} + \frac{\partial^2 U}{\partial z^2} - \frac{U}{r^2}\right) \\
U\frac{\partial V}{\partial r} + W\frac{\partial V}{\partial z} + \frac{UV}{r} 
&= + \nu \left( \frac{\partial^2 V}{\partial r^2} + \frac{1}{r}\frac{\partial V}{\partial r} + \frac{\partial^2 V}{\partial z^2} - \frac{V}{r^2}\right) \\
U\frac{\partial W}{\partial r} + W\frac{\partial W}{\partial z} 
&= - \frac{1}{\rho} \frac{\partial P}{\partial z} \nu \left( \frac{\partial^2 W}{\partial r^2} + \frac{1}{r}\frac{\partial W}{\partial r} + \frac{\partial^2 W}{\partial z^2} \right) \\
\frac{\partial U}{\partial r} + \frac{U}{r} + \frac{\partial W}{\partial z} &= 0 
\end{align*}$$

subject to the no-slip B.C. on the disk and B.C. at infinity

>$$ \begin{align*}
U = W = 0, V = r \Omega \quad \textrm{at } z = 0 \\
U = V = 0 \quad \textrm{at } z \rightarrow \infty
\end{align*}$$

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,"[ [OMP_PROC_BIND, spread] ]",\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("RotatingDiskBoundaryLayer", myBatch);

Project name is set to 'RotatingDiskBoundaryLayer'.
Creating database '\\dc3\userspace\smuda\hpccluster\RotatingDiskBoundaryLayer'.


## grid creation

In [ ]:
static public Grid3D RotatingDiskSector_CartesianCutOut(double radiusOP, double l_radial, double l_azimuthal, double h_axial, int res_radial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(radiusOP - (l_radial / 2.0), radiusOP + (l_radial / 2.0), res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 2.0, l_azimuthal / 2.0, res_azimuthal + 1);    // azimuthal direction
    // double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    double[] zNodes = GenericBlas.Logspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_CartesianCutOut_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_rotatingDisk");
    grd.EdgeTagNames.Add(2, "velocity_inlet_top");
    // grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    grd.EdgeTagNames.Add(3, "velocity_inlet_back");
    grd.EdgeTagNames.Add(4, "velocity_inlet_front");
    // grd.EdgeTagNames.Add(4, "pressure_outlet_front");
    grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(6, "velocity_inlet_downstream");
    // grd.EdgeTagNames.Add(6, "pressure_outlet_downstream");
    // grd.EdgeTagNames.Add(6, "outlet_rotDisk_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if (((X.x - radiusOP) + (l_radial / 2.0)).Abs() <= 1e-8)
            et = 3;
        if (((X.x - radiusOP) - (l_radial / 2.0)).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 6;

        return et;
    });    

    return grd;
}

## simulation setup

In [ ]:
double radiusOP = 0.09; //78e-3; // operating point -> Re = radiusOp / Lstar
double viscosity = 20.0e-6 / 1.2; //1.52e-5; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double omega = 38.0 / radiusOP; //1820.5128205128206; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(viscosity / omega);    // characteristic length scale
Lstar

0.00019867985355975657

In [ ]:
double reynolds = radiusOP / Lstar;
reynolds

452.990066116245

In [ ]:
double z = 5.5;
double zstar = z * Lstar;
zstar

0.001092739194578661

In [ ]:
double zTop = 10;
double zTopStar = zTop * Lstar;
zTopStar

0.0019867985355975656

In [ ]:
double density = 1.2; //1.19;

In [ ]:
omega

422.22222222222223

In [ ]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velX = - rOnRotDisk * Math.Sin(theta) * omega;" +
    "return velX; } "
);

In [ ]:
Formula RotatingDiskVelocityY = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velY = rOnRotDisk * Math.Cos(theta) * omega;" +
    "return velY; } "
);

### prescribed boundary conditions (initial conditions)

In [ ]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [ ]:
MultidimensionalArray velU_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_VelocityU.txt");
MultidimensionalArray velV_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_VelocityV.txt");
MultidimensionalArray velW_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_VelocityW.txt");
MultidimensionalArray pressureP_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_PressureP.txt");

In [ ]:
MultidimensionalArray velU_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_VelocityU.txt");
MultidimensionalArray velV_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_VelocityV.txt");
MultidimensionalArray velW_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_VelocityW.txt");
MultidimensionalArray pressureP_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_PressureP.txt");

In [ ]:
Plot2Ddata plt = new Plot2Ddata();
var fmtP = new PlotFormat();
fmtP.LineColor = LineColors.Blue;
var fmtM = new PlotFormat();

var fmtHAM = new PlotFormat();
fmtHAM.LineColor = LineColors.Green;

// var solP = velW_P;
// var solM = velW_M;
var solP = pressureP_P;
var solM = pressureP_M;

plt.AddDataGroup(solP.ExtractSubArrayShallow(new int[] { -1, 1}).To1DArray(), solP.ExtractSubArrayShallow(new int[] { -1, 0}).To1DArray(), fmtP);
plt.AddDataGroup(solM.ExtractSubArrayShallow(new int[] { -1, 1}).To1DArray(), solM.ExtractSubArrayShallow(new int[] { -1, 0}).To1DArray(), fmtM);
var gp = plt.ToGnuplot();
gp.PlotNow()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 -0.45 
 
 
 
 
 -0.4 
 
 
 
 
 -0.35 
 
 
 
 
 -0.3 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gnuplot_plot_1 
 
	<path stroke='rgb( 0, 0, 255)' d='M750.1,564.0 L748.4,564.0 L740.4,563.8 L732.6,563.7 L724.8,563.6 L717.1,563.4 L709.5,563.3 L701.9,563.2
 L694.5,563.0 L687.1,562.9 L679.7,562.8 L672.5,562.7 L665.3,562.5 L658.2,562.4 L651.2,562.3 L644.2,562.1
 L637.3,562.0 L630.5,561.9 L623.8,561.7 L617.1,561.6 L610.5,561.5 L603.9,561.3 L597.5,561.2 L591.1,561.1
 L584.7,560.9 L578.4,560.8 L572.2,560.7 L566.1,560.5 L560.0,560.4 L554.0,560.3 L548.1,560.1 L542.2,560.0
 L536.4,559.9 L530.6,559.7 L525.0,559.6 L519.3,559.5 L513.8,559.4 L508.3,559.2 L502.8,559.1 L497.4,559.0
 L492.1,558.8 L486.8,558.7 L481.6,558.6 L476.5,558.4 L471.4,558.3 L466.4,558.2 L461.4,558.0 L456.5,557.9
 L451.6,557.8 L446.8,557.6 L442.0,557.5 L437.3,557.4 L432.7,557.2 L428.1,557.1 L423.5,557.0 L419.0,556.8
 L414.6,556.7 L410.2,556.6 L405.9,556.4 L401.6,556.3 L397.4,556.2 L393.2,556.1 L389.1,555.9 L385.0,555.8
 L380.9,555.7 L377.0,555.5 L373.0,555.4 L369.1,555.3 L365.3,555.1 L361.5,555.0 L357.7,554.9 L354.0,554.7
 L350.4,554.6 L346.7,554.5 L343.2,554.3 L339.6,554.2 L336.1,554.1 L332.7,553.9 L329.3,553.8 L325.9,553.7
 L322.6,553.5 L319.3,553.4 L316.1,553.3 L312.9,553.1 L309.8,553.0 L306.7,552.9 L303.6,552.8 L300.5,552.6
 L297.5,552.5 L294.6,552.4 L291.7,552.2 L288.8,552.1 L285.9,552.0 L283.1,551.8 L280.4,551.7 L277.6,551.6
 L274.9,551.4 L272.3,551.3 L269.6,551.2 L267.0,551.0 L264.5,550.9 L261.9,550.8 L259.4,550.6 L257.0,550.5
 L254.6,550.4 L252.2,550.2 L249.8,550.1 L247.5,550.0 L245.2,549.8 L242.9,549.7 L240.7,549.6 L238.4,549.5
 L236.3,549.3 L234.1,549.2 L232.0,549.1 L229.9,548.9 L227.9,548.8 L225.8,548.7 L223.8,548.5 L221.8,548.4
 L219.9,548.3 L218.0,548.1 L216.1,548.0 L214.2,547.9 L212.4,547.7 L210.6,547.6 L208.8,547.5 L207.0,547.3
 L205.3,547.2 L203.6,547.1 L201.9,546.9 L200.2,546.8 L198.6,546.7 L197.0,546.6 L195.4,546.4 L193.8,546.3
 L192.3,546.2 L190.8,546.0 L189.3,545.9 L187.8,545.8 L186.3,545.6 L184.9,545.5 L183.5,545.4 L182.1,545.2
 L180.7,545.1 L179.4,545.0 L178.1,544.8 L176.8,544.7 L175.5,544.6 L174.2,544.4 L173.0,544.3 L171.8,544.2
 L170.5,544.0 L169.4,543.9 L168.2,543.8 L167.0,543.6 L165.9,543.5 L164.8,543.4 L163.7,543.3 L162.6,543.1
 L161.6,543.0 L160.5,542.9 L159.5,542.7 L158.5,542.6 L157.5,542.5 L156.5,542.3 L155.6,542.2 L154.6,542.1
 L153.7,541.9 L152.8,541.8 L151.9,541.7 L151.0,541.5 L150.1,541.4 L149.3,541.3 L148.4,541.1 L147.6,541.0
 L146.8,540.9 L146.0,540.7 L145.2,540.6 L144.5,540.5 L143.7,540.3 L143.0,540.2 L142.2,540.1 L141.5,540.0
 L140.8,539.8 L140.1,539.7 L139.5,539.6 L138.8,539.4 L138.2,539.3 L137.5,539.2 L136.9,539.0 L136.3,538.9
 L135.7,538.8 L135.1,538.6 L134.5,538.5 L133.9,538.4 L133.4,538.2 L132.8,538.1 L132.3,538.0 L131.8,537.8
 L131.2,537.7 L130.7,537.6 L130.2,537.4 L129.8,537.3 L129.3,537.2 L128.8,537.0 L128.4,536.9 L127.9,536.8
 L127.5,536.7 L127.0,536.5 L126.6,536.4 L126.2,536.3 L125.8,536.1 L125.4,536.0 L125.0,535.9 L124.7,535.7
 L124.3,535.6 L123.9,535.5 L123.6,535.3 L123.2,535.2 L122.9,535.1 L122.6,534.9 L122.3,534.8 L121.9,534.7
 L121.6,534.5 L121.3,534.4 L121.0,534.3 L120.8,534.1 L120.5,534.0 L120.2,533.9 L120.0,533.7 L119.7,533.6
 L119.4,533.5 L119.2,533.4 L119.0,533.2 L118.7,533.1 L118.5,533.0 L118.3,532.8 L118.1,532.7 L117.9,532.6
 L117.7,532.4 L117.5,532.3 L117.3,532.2 L117.1,532.0 L116.9,531.9 L116.8,531.8 L116.6,531.6 L116.4,531.5
 L116.3,531.4 L116.1,531.2 L116.0,531.1 L115.

In [ ]:
var velU = velU_P;
var velV = velV_P;
var velW = velW_P;
var pressureP = pressureP_P;

double[] zOrgValues = velU.GetColumn(0);
double[][] orgValues = new double[4][];
orgValues[0] = velU.GetColumn(1);
orgValues[1] = velV.GetColumn(1);
orgValues[2] = velW.GetColumn(1);
orgValues[3] = pressureP.GetColumn(1);

var vonKarman_velX = new RotatingDiskBoundaryLayerConditions() { selectedEvalFlowField = evalFlowFields.velocityX };
vonKarman_velX.SetData(zOrgValues, velU.GetColumn(1), velV.GetColumn(1), velW.GetColumn(1), pressureP.GetColumn(1), omega, viscosity, density);
var vonKarman_velY = new RotatingDiskBoundaryLayerConditions() { selectedEvalFlowField = evalFlowFields.velocityY };
vonKarman_velY.SetData(zOrgValues, velU.GetColumn(1), velV.GetColumn(1), velW.GetColumn(1), pressureP.GetColumn(1), omega, viscosity, density);
var vonKarman_velZ = new RotatingDiskBoundaryLayerConditions() { selectedEvalFlowField = evalFlowFields.velocityZ };
vonKarman_velZ.SetData(zOrgValues, velU.GetColumn(1), velV.GetColumn(1), velW.GetColumn(1), pressureP.GetColumn(1), omega, viscosity, density);
var vonKarman_P = new RotatingDiskBoundaryLayerConditions() { selectedEvalFlowField = evalFlowFields.pressureP };
vonKarman_P.SetData(zOrgValues, velU.GetColumn(1), velV.GetColumn(1), velW.GetColumn(1), pressureP.GetColumn(1), omega, viscosity, density);

In [ ]:
// MultidimensionalArray HAMcoeff_velU = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffU.txt");
// MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");
// MultidimensionalArray HAMcoeff_velW = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffW.txt");
// MultidimensionalArray HAMcoeff_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffP.txt");

MultidimensionalArray HAMcoeff_velU = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAM30coeffU.txt");
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAM30coeffV.txt");
MultidimensionalArray HAMcoeff_velW = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAM30coeffW.txt");
MultidimensionalArray HAMcoeff_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAM30coeffP.txt");

In [ ]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX();
vonKarmanHAM_velX.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, viscosity, density);
var vonKarmanHAM_velY = new RotatingDiskBoundaryLayerConditionsHAM_VelocityY();
vonKarmanHAM_velY.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, viscosity, density);
var vonKarmanHAM_velZ = new RotatingDiskBoundaryLayerConditionsHAM_VelocityZ();
vonKarmanHAM_velZ.SetData(HAMcoeff_velW, omega, viscosity, density);
var vonKarmanHAM_P = new RotatingDiskBoundaryLayerConditionsHAM_PressureP();
vonKarmanHAM_P.SetData(HAMcoeff_P, omega, viscosity, density);

In [ ]:
// var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityU();
// vonKarmanHAM_velX.SetData(HAMcoeff_velU, omega, viscosity, density);
// var vonKarmanHAM_velY = new RotatingDiskBoundaryLayerConditionsHAM_VelocityV();
// vonKarmanHAM_velY.SetData(HAMcoeff_velV, omega, viscosity, density);

In [ ]:
double[] zStarValues = zOrgValues.CloneAs();
zStarValues.ScaleV(Lstar);
int numOrgVal = zStarValues.Length;
double[][] varValues = new double[4][];
varValues[0] = new double[numOrgVal];
varValues[1] = new double[numOrgVal];
varValues[2] = new double[numOrgVal];
varValues[3] = new double[numOrgVal];

double[] evalPoint = new double[] {radiusOP - 2.0e-4, +4.0e-4};
double rStarOnDisk = Math.Sqrt(evalPoint[0].Pow2() + evalPoint[1].Pow2()); 
double theta = Math.Atan2(evalPoint[1], evalPoint[0]);

string[] varNames = new string[] {"VelocityX", "VelocityY", "VelocityZ", "Pressure"};
for (int i = 0; i < numOrgVal; i++) {
    // double velX = vonKarman_velX.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    // double velY = vonKarman_velY.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    // varValues[0][i] = (velX * Math.Cos(theta) + velY * Math.Sin(theta)) / (rStarOnDisk * omega); 
    // varValues[1][i] = (velX * Math.Sin(theta) + velY * Math.Cos(theta)) / (rStarOnDisk * omega); 
    // varValues[2][i] = vonKarman_velZ.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0) / Math.Sqrt(viscosity * omega);
    // varValues[3][i] = vonKarman_P.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0) / (density * viscosity * omega);
    double velX = vonKarmanHAM_velX.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    double velY = vonKarmanHAM_velY.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[0][i] = (velX * Math.Cos(theta) + velY * Math.Sin(theta)) / (rStarOnDisk * omega); 
    varValues[1][i] = (velX * Math.Sin(theta) + velY * Math.Cos(theta)) / (rStarOnDisk * omega); 
    varValues[2][i] = vonKarmanHAM_velZ.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0) / Math.Sqrt(viscosity * omega);
    varValues[3][i] = vonKarmanHAM_P.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0) / (density * viscosity * omega);
}

In [ ]:
Plot2Ddata plt = new Plot2Ddata();
var fmtOrg = new PlotFormat();
fmtOrg.LineColor = LineColors.Black;
fmtOrg.LineWidth = 2;
var fmt = new PlotFormat();
fmt.DashType = DashTypes.Dashed;
fmt.LineWidth = 2;

int idx = 3;
plt.AddDataGroup(varNames[idx], orgValues[idx], zOrgValues, fmtOrg);
plt.AddDataGroup(varNames[idx], varValues[idx], zOrgValues, fmt);
var gp = plt.ToGnuplot();
gp.PlotNow()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 -0.45 
 
 
 
 
 -0.4 
 
 
 
 
 -0.35 
 
 
 
 
 -0.3 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Pressure 
 
 
 
 
 Pressure 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M588.5,57.1 L641.9,57.1 M750.1,564.0 L748.4,564.0 L740.4,563.8 L732.6,563.7 L724.8,563.6 L717.1,563.4
 L709.5,563.3 L701.9,563.2 L694.5,563.0 L687.1,562.9 L679.7,562.8 L672.5,562.7 L665.3,562.5 L658.2,562.4
 L651.2,562.3 L644.2,562.1 L637.3,562.0 L630.5,561.9 L623.8,561.7 L617.1,561.6 L610.5,561.5 L603.9,561.3
 L597.5,561.2 L591.1,561.1 L584.7,560.9 L578.4,560.8 L572.2,560.7 L566.1,560.5 L560.0,560.4 L554.0,560.3
 L548.1,560.1 L542.2,560.0 L536.4,559.9 L530.6,559.7 L525.0,559.6 L519.3,559.5 L513.8,559.4 L508.3,559.2
 L502.8,559.1 L497.4,559.0 L492.1,558.8 L486.8,558.7 L481.6,558.6 L476.5,558.4 L471.4,558.3 L466.4,558.2
 L461.4,558.0 L456.5,557.9 L451.6,557.8 L446.8,557.6 L442.0,557.5 L437.3,557.4 L432.7,557.2 L428.1,557.1
 L423.5,557.0 L419.0,556.8 L414.6,556.7 L410.2,556.6 L405.9,556.4 L401.6,556.3 L397.4,556.2 L393.2,556.1
 L389.1,555.9 L385.0,555.8 L380.9,555.7 L377.0,555.5 L373.0,555.4 L369.1,555.3 L365.3,555.1 L361.5,555.0
 L357.7,554.9 L354.0,554.7 L350.4,554.6 L346.7,554.5 L343.2,554.3 L339.6,554.2 L336.1,554.1 L332.7,553.9
 L329.3,553.8 L325.9,553.7 L322.6,553.5 L319.3,553.4 L316.1,553.3 L312.9,553.1 L309.8,553.0 L306.7,552.9
 L303.6,552.8 L300.5,552.6 L297.5,552.5 L294.6,552.4 L291.7,552.2 L288.8,552.1 L285.9,552.0 L283.1,551.8
 L280.4,551.7 L277.6,551.6 L274.9,551.4 L272.3,551.3 L269.6,551.2 L267.0,551.0 L264.5,550.9 L261.9,550.8
 L259.4,550.6 L257.0,550.5 L254.6,550.4 L252.2,550.2 L249.8,550.1 L247.5,550.0 L245.2,549.8 L242.9,549.7
 L240.7,549.6 L238.4,549.5 L236.3,549.3 L234.1,549.2 L232.0,549.1 L229.9,548.9 L227.9,548.8 L225.8,548.7
 L223.8,548.5 L221.8,548.4 L219.9,548.3 L218.0,548.1 L216.1,548.0 L214.2,547.9 L212.4,547.7 L210.6,547.6
 L208.8,547.5 L207.0,547.3 L205.3,547.2 L203.6,547.1 L201.9,546.9 L200.2,546.8 L198.6,546.7 L197.0,546.6
 L195.4,546.4 L193.8,546.3 L192.3,546.2 L190.8,546.0 L189.3,545.9 L187.8,545.8 L186.3,545.6 L184.9,545.5
 L183.5,545.4 L182.1,545.2 L180.7,545.1 L179.4,545.0 L178.1,544.8 L176.8,544.7 L175.5,544.6 L174.2,544.4
 L173.0,544.3 L171.8,544.2 L170.5,544.0 L169.4,543.9 L168.2,543.8 L167.0,543.6 L165.9,543.5 L164.8,543.4
 L163.7,543.3 L162.6,543.1 L161.6,543.0 L160.5,542.9 L159.5,542.7 L158.5,542.6 L157.5,542.5 L156.5,542.3
 L155.6,542.2 L154.6,542.1 L153.7,541.9 L152.8,541.8 L151.9,541.7 L151.0,541.5 L150.1,541.4 L149.3,541.3
 L148.4,541.1 L147.6,541.0 L146.8,540.9 L146.0,540.7 L145.2,540.6 L144.5,540.5 L143.7,540.3 L143.0,540.2
 L142.2,540.1 L141.5,540.0 L140.8,539.8 L140.1,539.7 L139.5,539.6 L138.8,539.4 L138.2,539.3 L137.5,539.2
 L136.9,539.0 L136.3,538.9 L135.7,538.8 L135.1,538.6 L134.5,538.5 L133.9,538.4 L133.4,538.2 L132.8,538.1
 L132.3,538.0 L131.8,537.8 L131.2,537.7 L130.7,537.6 L130.2,537.4 L129.8,537.3 L129.3,537.2 L128.8,537.0
 L128.4,536.9 L127.9,536.8 L127.5,536.7 L127.0,536.5 L126.6,536.4 L126.2,536.3 L125.8,536.1 L125.4,536.0
 L125.0,535.9 L124.7,535.7 L124.3,535.6 L123.9,535.5 L123.6,535.3 L123.2,535.2 L122.9,535.1 L122.6,534.9
 L122.3,534.8 L121.9,534.7 L121.6,534.5 L121.3,534.4 L121.0,534.3 L120.8,534.1 L120.5,534.0 L120.2,533.9
 L120.0,533.7 L119.7,533.6 L119.4,533.5 L119.2,533.4 L119.0,533.2 L118.7,533.1 L118.5,533.0 L118.3,532.8
 L118.1,532.7 L117.9,532.6 L117.7,532.4 L117.5,532.3 L117.3,532.2 L117.1,532.0 L116.9,531.9 L116.8,531.8
 L116.6,531.6 L116.4,531.5 L116.

In [ ]:
double velW_top = -0.88447342;
double velWstar_top = velW_top * Math.Sqrt(viscosity * omega); 
velWstar_top

-0.07419586537108543

In [ ]:
Formula VelocityZ_top = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.07419586537108543; } "
);

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();

int k = 3;
C.SetDGdegree(k);


// physical parameters
C.PhysicalParameters.rho_A = density;
C.PhysicalParameters.mu_A = density * viscosity;

C.PhysicalParameters.IncludeConvection = true;

    
// set grid
double l_radial = zTopStar / 2.0; 
double l_azimuthal = zTopStar / 2.0;
double h_axial = zTopStar; 

int res_global = 8;
int res_radial = 1 * res_global; 
int res_azimuthal = 1 * res_global;
int res_axial = 2 * res_global;

Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial, l_azimuthal, h_axial, res_radial, res_azimuthal, res_axial);
C.SetGrid(grd);

// C.UseRotInertForceTerms = true;
// C.AngularVelocity = new double[] { 0.0, 0.0, omega };
// C.UseCylindricalCoords = true;

// boundary conditions
// C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#A", RotatingDiskVelocityX);
// C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityY#A", RotatingDiskVelocityY);
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityZ#A", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_top", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_top", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_top", "VelocityZ#A", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_front", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_front", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_front", "VelocityZ#A", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_back", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_back", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_back", "VelocityZ#A", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityZ#A", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_downstream", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_downstream", "VelocityZ#A", vonKarmanHAM_velZ);
// C.AddBoundaryValue("pressure_dirichlet_downstream", "Pressure#A", vonKarman_P);


// initial conditions
C.AddInitialValue("VelocityX#A", vonKarmanHAM_velX);
C.AddInitialValue("VelocityY#A", vonKarmanHAM_velY);
C.AddInitialValue("VelocityZ#A", vonKarmanHAM_velZ);
C.AddInitialValue("Pressure#A", vonKarmanHAM_P);


C.SkipSolveAndEvaluateResidual = true;


C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
// C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();

// C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
// C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
// C.dtFixed = Math.PI * 1.0e-3;
// C.NoOfTimesteps = 2000;
    
// {
//     C.AdaptiveMeshRefinement = true;
//     int AMRlevel = 1;
//     C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel });
//     C.AMR_startUpSweeps = AMRlevel;
// }
    
C.SessionName = "RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16log";
    
Controls.Add(C);

Error: System.ApplicationException: grid corrupted: contains NAN - value.
   at BoSSS.Foundation.Grid.Classic.GridCommons.CheckGridForNANorINF() in D:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\GridCommons.cs:line 997
   at BoSSS.Foundation.Grid.Classic.GridData..ctor(GridCommons grd) in D:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\GridData.cs:line 177
   at BoSSS.Foundation.Grid.Classic.GridCommons.InitGridData() in D:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\GridCommons.cs:line 1159
   at BoSSS.Foundation.Grid.Classic.GridCommons.get_GridData() in D:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\GridCommons.cs:line 1113
   at BoSSS.Foundation.Grid.Classic.GridCommons.get_iGridData() in D:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\GridCommons.cs:line 1104
   at BoSSS.Foundation.Grid.IGrid_Extensions.DefineEdgeTags(IGrid g, Func`2 EdgeTagFunc, String[] EdgeTagNamesToEnsure) in D:\BoSSS-expe

## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

(empty)

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    oneJob.NumberOfMPIProcs = 1;
    oneJob.Activate(myBatch); 
}

## Postprocessing

In [ ]:
BoSSSshell.WorkflowMgm.Sessions

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\RotatingDiskBoundaryLayer");

Opening existing database '\\hpccluster\hpccluster-scratch\smuda\RotatingDiskBoundaryLayer'.


In [ ]:
databases.Pick(0).Sessions

#0: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16amr1_pressureOutlet	08/31/2023 11:22:57	e394ce5a...
#1: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16amr1_pressureOutlet2	08/31/2023 11:41:27	b069de64...
#2: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16amr1_stratified	08/31/2023 12:21:17	1c90cffa...
#3: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16amr1_4	08/31/2023 11:08:20	eb023bc9...
#4: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16amr1_3	08/31/2023 10:58:49	828a8a22...
#5: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_8x8x16amr1_2	08/30/2023 13:23:28	5bb46b38...
#6: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_32x32x64amr2	08/25/2023 10:11:56	3776e9a7...
#7: RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_32x32x64amr1	08/25/2023 08:56:52	a35ef211...
#8: RotatingDiskBoundaryLaye

In [ ]:
//var sess = BoSSSshell.WorkflowMgm.Sessions.Pick(0);
var sess = databases.Pick(0).Sessions.Pick(6);
sess

RotatingDiskBoundaryLayer	RotatingDiskBoundaryLayer-TestRunRe100HAM_32x32x64amr2	08/25/2023 10:11:56	3776e9a7...

In [ ]:
sess.KeysAndQueries["SkipSolveAndEvaluateResidual"]

True

In [ ]:
sess.Timesteps

#0:  { Time-step: 0.0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, MPIrank, CutCells; Name:  }
#1:  { Time-step: 0.1; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, MPIrank, CutCells; Name:  }
#2:  { Time-step: 0.2; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, MPIrank, CutCells; Name:  }
#3:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#4:  { Time-step: 1; Physical time: 1.7976931348623158E+304s; Fields: Phi, PhiDG, VelocityX, VelocityY, Vel

In [ ]:
sess.Timesteps.Skip(4).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RotatingDiskBoundaryLayer__RotatingDiskBoundaryLayer-TestRunRe100HAM_32x32x64amr2__3776e9a7-6941-4622-b414-e77cb4b9faae


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RotatingDiskBoundaryLayer__RotatingDiskBoundaryLayer-TestRunRe100HAM_32x32x64amr2__3776e9a7-6941-4622-b414-e77cb4b9faae

### velocity profiles

In [ ]:
int tIdx = 2;
sess.Timesteps.Pick(tIdx)

 { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }

In [ ]:
DGField[] solutionFields = new DGField[4];
solutionFields[0] = sess.Timesteps.Pick(tIdx).GetField("VelocityX");
solutionFields[1] = sess.Timesteps.Pick(tIdx).GetField("VelocityY");
solutionFields[2] = sess.Timesteps.Pick(tIdx).GetField("VelocityZ");
solutionFields[3] = sess.Timesteps.Pick(tIdx).GetField("Pressure");

In [ ]:
zTopStar

0.0009137448098155133

In [ ]:
double[][] solutionValues = new double[4][];
double[] evalPoint = new double[] {radiusOP - 1.0e-5, zTopStar / 4.0};

int numVal = 100;
double stepSize = 0.0001 / numVal; //zTopStar / numVal; 
double[] zStarValues = Enumerable.Range(0, numVal+1).Select(idx => idx * stepSize).ToArray();

// velocity fields
for (int j = 0; j < 3; j++) {
    solutionValues[j] = new double[numVal+1];
    for (int i = 0; i <= numVal; i++) {
        try {
            solutionValues[j][i] = solutionFields[j].ProbeAt(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]});
        }
        catch {
            Console.WriteLine("evalPoint = ( {0}, {1}, {2} ) not found", evalPoint[0], evalPoint[1], zStarValues[i]);
        }
    }
}
// pressure field (correct to same pressure level at wall p=0)
solutionValues[3] = new double[numVal+1];
double pressure0 = 0.0; //solutionFields[3].ProbeAt(new double[] {evalPoint[0], evalPoint[1], 0.0});
for (int i = 0; i <= numVal; i++) {
    try {
        solutionValues[3][i] = solutionFields[3].ProbeAt(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}) - pressure0;
    }
    catch {
        Console.WriteLine("evalPoint = ( {0}, {1}, {2} ) not found", evalPoint[0], evalPoint[1], zStarValues[i]);
    }
}

In [ ]:
pressure0

-0.0009042693351911418

In [ ]:
double[][] varValues = new double[4][];
varValues[0] = new double[numVal+1];
varValues[1] = new double[numVal+1];
varValues[2] = new double[numVal+1];
varValues[3] = new double[numVal+1];

for (int i = 0; i <= numVal; i++) {
    varValues[0][i] = vonKarman_velX.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[1][i] = vonKarman_velY.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0); 
    varValues[2][i] = vonKarman_velZ.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[3][i] = vonKarman_P.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
}

In [ ]:
var fmt1 = new PlotFormat();    // blue - BoSSS
fmt1.LineColor = LineColors.Blue;
var fmt2 = new PlotFormat();    // red - Analytic

var gp = new Gnuplot();
gp.SetMultiplot(2,2);

int idx = 0;
for (int i = 0; i < 2; i++)
for (int j = 0; j < 2; j++) {
    Plot2Ddata plt = new Plot2Ddata(); 
    plt.AddDataGroup("BoSSS", solutionValues[idx], zStarValues, fmt1);
    plt.AddDataGroup("Analytic", varValues[idx], zStarValues, fmt2);
    gp.SetSubPlot(i,j, varNames[idx]);
    plt.ToGnuplot(gp);
    idx++;
}
//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:1000)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 1x10 -5 
 
 
 
 
 2x10 -5 
 
 
 
 
 3x10 -5 
 
 
 
 
 4x10 -5 
 
 
 
 
 5x10 -5 
 
 
 
 
 6x10 -5 
 
 
 
 
 7x10 -5 
 
 
 
 
 8x10 -5 
 
 
 
 
 9x10 -5 
 
 
 
 
 0.0001 
 
 
 
 
 -5 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 25 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 BoSSS 
 
 
 BoSSS 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M531.0,62.1 L584.4,62.1 M165.6,459.1 L179.3,454.9 L192.6,450.7 L205.7,446.6 L218.4,442.4 L230.9,438.2
 L243.2,434.0 L255.1,429.8 L266.8,425.7 L278.2,421.5 L289.4,417.3 L300.3,413.1 L310.9,408.9 L321.3,404.8
 L331.5,400.6 L341.4,396.4 L351.1,392.2 L360.5,388.0 L369.7,383.9 L378.7,379.7 L387.5,375.5 L396.0,371.3
 L404.3,367.1 L412.5,363.0 L420.3,358.8 L428.0,354.6 L435.5,350.4 L442.8,346.2 L449.9,342.1 L456.8,337.9
 L463.5,333.7 L470.0,329.5 L476.3,325.3 L482.4,321.2 L488.4,317.0 L494.2,312.8 L499.8,308.6 L505.2,304.4
 L510.5,300.3 L515.6,296.1 L520.5,291.9 L525.3,287.7 L529.9,283.5 L534.4,279.4 L538.7,275.2 L542.8,271.0
 L546.9,266.8 L550.7,262.6 L554.5,258.5 L558.1,254.3 L561.5,250.1 L564.9,245.9 L568.1,241.7 L571.1,237.6
 L574.1,233.4 L576.9,229.2 L579.6,225.0 L582.1,220.8 L584.6,216.7 L586.9,212.5 L589.2,208.3 L591.3,204.1
 L593.3,199.9 L595.2,195.8 L597.0,191.6 L598.7,187.4 L600.2,183.2 L601.7,179.0 L603.1,174.9 L604.4,170.7
 L605.6,166.5 L606.7,162.3 L607.8,158.1 L608.7,154.0 L609.6,149.8 L610.4,145.6 L611.1,141.4 L611.7,137.2
 L612.2,133.1 L612.6,128.9 L613.0,124.7 L613.3,120.5 L613.6,116.3 L613.7,112.2 L613.8,108.0 L613.8,103.8
 L613.8,99.6 L613.7,95.4 L613.6,91.3 L613.3,87.1 L613.1,82.9 L612.7,78.7 L612.3,74.5 L611.9,70.4
 L611.4,66.2 L610.8,62.0 L610.2,57.8 L609.5,53.6 L608.8,49.5 L608.0,45.3 L607.2,41.1 '/> 
 
 Analytic 
 
 
 Analytic 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M531.0,86.1 L584.4,86.1 M165.6,459.1 L179.3,454.9 L192.6,450.7 L205.7,446.6 L218.4,442.4 L230.9,438.2
 L243.2,434.0 L255.1,429.8 L266.8,425.7 L278.2,421.5 L289.4,417.3 L300.3,413.1 L310.9,408.9 L321.3,404.8
 L331.5,400.6 L341.4,396.4 L351.1,392.2 L360.5,388.0 L369.7,383.9 L378.7,379.7 L387.5,375.5 L396.0,371.3
 L404.3,367.1 L412.4,363.0 L420.3,358.8 L428.0,354.6 L435.5,350.4 L442.8,346.2 L449.9,342.1 L456.8,337.9
 L463.5,333.7 L470.0,329.5 L476.3,325.3 L482.4,321.2 L488.4,317.0 L494.2,312.8 L499.8,308.6 L505.2,304.4
 L510.5,300.3 L515.6,296.1 L520.5,291.9 L525.3,287.7 L529.9,283.5 L534.4,279.4 L538.7,275.2 L542.9,271.0
 L546.9,266.8 L550.7,262.6 L554.5,258.5 L558.1,254.3 L561.5,250.1 L564.9,245.9 L568.1,241.7 L571.1,237.6
 L574.1,233.4 L576.9,229.2 L579.6,225.0 L582.1,220.8 L584.6,216.7 L586.9,212.5 L589.2,208.3 L591.3,204.1
 L593.3,199.9 L595.2,195.8 L597.0,191.6 L598.7,187.4 L600.2,183.2 L601.7,179.0 L603.1,174.9 L604.4,170.7
 L605.6,166.5 L606.8,162.3 L607.8,158.1 L608.7,154.0 L609.6,149.8 L610.4,145.6 L611.0,141.4 L611.7,137.2
 L612.2,133.1 L612.6,128.9 L613.0,124.7 L613.3,120.5 L613.6,116.3 L613.7,112.2 L613.8,108.0 L613.9,103.8
 L613.8,99.6 L613.7,95.4 L613.6,91.3 L613.3,87.1 L613.1,82.9 L612.7,78.7 L612.3,74.5 L611.9,70.4
 L611.3,66.2 L610.8,62.0 L610.2,57.8 L609.5,53.6 L608.8,49.5 L608.0,45.3 L607.2,41.1 '/> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 1x10 -5 
 
 
 
 
 2x10 -5 
 
 
 
 
 3x10 -5 
 
 
 
 
 4x10 -5 
 
 
 
 
 5x10 -5 
 
 
 
 
 6x10 -5 
 
 
 
 
 7x10 -5 
 
 
 
 
 8x10 -5 
 
 
 
 
 9x10 -5 
 
 
 
 
 0.0001 
 
 
 
 
 60 
 
 
 
 
 70 
 
 
 
 
 80 
 
 
 
 
 90 
 
 
 
 
 100 
 
 
 
 
 110 
 
 
 
 
 120 
 
 
 
 
 130 
 
 
 
 
 140 
 
 
 
 
 150 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

### mass conservation

In [ ]:
using BoSSS.Foundation.Quadrature;

In [ ]:
int tIdx = 2;
sess.Timesteps.Pick(tIdx)

 { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }

In [ ]:
int D = 3;
DGField[] velocityFields = new DGField[D];
velocityFields[0] = ((XDGField)sess.Timesteps.Pick(tIdx).GetField("VelocityX")).GetSpeciesShadowField("A");
velocityFields[1] = ((XDGField)sess.Timesteps.Pick(tIdx).GetField("VelocityY")).GetSpeciesShadowField("A");
velocityFields[2] = ((XDGField)sess.Timesteps.Pick(tIdx).GetField("VelocityZ")).GetSpeciesShadowField("A");

In [ ]:
int degU = velocityFields[0].Basis.Degree;
var grdDat = velocityFields[0].GridDat;

In [ ]:
double GetMassFluxAtBoundary(IGridData grdDat, DGField[] velocityFields, int degU, int EdgeTag) {

    EdgeMask boundaryMask; 
    if (EdgeTag > 0 && EdgeTag < 7) {
        int numE = grdDat.iGeomEdges.Count;
        BitArray BoundaryMaskBA = new BitArray(numE);
        for(int eIdx = 0; eIdx < numE; eIdx++) {
            if (grdDat.iGeomEdges.EdgeTags[eIdx] == EdgeTag)
                BoundaryMaskBA[eIdx] = true;
        }
        boundaryMask = new EdgeMask(grdDat, BoundaryMaskBA);
    } else {
        boundaryMask = grdDat.GetBoundaryEdgeMask();
    }

    var eqs = new EdgeQuadratureScheme(domain:boundaryMask);

    double massFluxBoundary = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, grdDat,
        eqs.Compile(grdDat, 2*degU),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {

            var normals = grdDat.iGeomEdges.NormalsCache.GetNormals_Edge(QR.Nodes, i0, Length);
            int K = EvalResult.GetLength(1);
            for (int d = 0; d < D; d++) {
                var ValueIn = MultidimensionalArray.Create(new int[] { Length, K});
                var ValueOut = MultidimensionalArray.Create(new int[] { Length, K});
                velocityFields[d].EvaluateEdge(i0, Length, QR.Nodes, ValueIn, ValueOut, null, null, null, null, 0, 1.0);

                for (int j = 0; j < Length; j++) {
                    for (int k = 0; k < K; k++) {
                        EvalResult[j, k, 0] += ValueIn[j, k] * normals[j, k, d];
                    }
                }
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for (int i = 0; i < Length; i++)
            massFluxBoundary += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    return massFluxBoundary;
}

In [ ]:
// boundary edge mask
GetMassFluxAtBoundary(grdDat, velocityFields, degU, -1)

-4.2993604988409033E-13

In [ ]:
// edgeTag 1 - rotating Disk 
double mfb1 = GetMassFluxAtBoundary(grdDat, velocityFields, degU, 1);
mfb1

-3.1866268930222583E-13

In [ ]:
// edgeTag 2 - top
double mfb2 = GetMassFluxAtBoundary(grdDat, velocityFields, degU, 2);
mfb2

-3.0700517847690185E-08

In [ ]:
double massFluxTop = vonKarman_velZ.Evaluate(new double[] {radiusOP, 0.0, zTopStar}, 0.0) * (0.5 * zTopStar) * (0.5 * zTopStar);
(mfb2 - massFluxTop) // massFluxTop

-1.0363022819707504E-14

In [ ]:
// edgeTag 3 - back
double mfb3 = GetMassFluxAtBoundary(grdDat, velocityFields, degU, 3);
mfb3

-2.6130037652277377E-06

In [ ]:
// edgeTag 4 - front
double mfb4 = GetMassFluxAtBoundary(grdDat, velocityFields, degU, 4);
mfb4

2.628353968514896E-06

In [ ]:
// edgeTag 5 - upstream 
double mfb5 = GetMassFluxAtBoundary(grdDat, velocityFields, degU, 5);
mfb5

-7.528255132990838E-06

In [ ]:
// edgeTag 6 - downstream 
double mfb6 = GetMassFluxAtBoundary(grdDat, velocityFields, degU, 6);
mfb6

7.543605336277999E-06

In [ ]:
mfb5 + mfb6

1.5350203287160882E-08